# Instagram two-agent pipeline (Agent1 unchanged) + Tavily evidence + improved Agent2

- Agent1: splits text into **Discriminative Text** and **Residual Text** (kept identical to your previous logic).
- Tavily: searches for **contact/company legitimacy + scam reports** for discriminative samples.
- Agent2: makes final label using **(a) agent1 output + (b) Tavily evidence**.

> **Label convention used here (based on your `instagram_final_results` errors):**
> - **1 = benign / normal profile**
> - **0 = suspicious / potential scam / unverifiable contact**

Set environment variables before running:
- `OPENAI_API_KEY`, `OPENAI_BASE_URL` (optional), `TAVILY_API_KEY`


In [ ]:
# -*- coding: utf-8 -*-

import os
import re
import time
import json
import torch
import requests
import pandas as pd
from typing import List, Dict, Any
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1) Config & Keys

HF_TOKEN = "Input your key"
TAVILY_API_KEY = "Input your key"
TAVILY_ENDPOINT = "Input your URL"

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"


# 2) Load Local Model

print(f"Loading local model: {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN
)
print("Model loaded successfully.")


# 3) Inference & Utility Functions

def local_llama_generate(user_prompt: str, temperature: float = 0.0, max_tokens: int = 500) -> str:
    """Run local inference using HF transformers."""
    messages = [{"role": "user", "content": user_prompt}]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    do_sample = temperature > 0.0

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature if do_sample else None,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response

def retry_tavily(func, *args, retries=5, delay=2, **kwargs):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_err = e
            time.sleep(delay)
            delay *= 2
    raise last_err


# 4) Agent 1: Text Splitter

AGENT1_PROMPT = r"""
You are a text splitter/classifier.
TASK
Split the provided input text into TWO buckets:
1) Discriminative Text
2) Residual Text
NON-NEGOTIABLE RULES (VERY IMPORTANT)
- Do NOT add any commentary, explanation, bullets, or extra headings.
- Do NOT paraphrase, summarize, translate, rewrite, or normalize.
- Do NOT delete anything: every line from the input MUST appear exactly once
  in either Discriminative Text or Residual Text.
- Preserve the original order of lines exactly as in the input.
- Keep emojis and special characters exactly as-is.
- If you are unsure, put it into Residual Text (but still do not delete it).
HARD DISCRIMINATIVE RULES (must go to Discriminative Text)
If a line contains ANY of the following, it MUST be Discriminative Text:
- '@' (email address or social handle, e.g., name@gmail.com, @username)
- '#' (hashtag/topic tag, e.g., #SEO, #London, #Trending, #Crypto)
- A URL or domain (http, https, www, .com, .co, .uk, .net, linktr.ee, beacons.ai, etc.)
- Contact / collaboration / business signals:
  "DM", "inquiries", "enquiries", "contact", "email", "collab", "collaborate",
  "booking", "bookings", "manager", "managed by", "agent", "PR", "press",
  "WhatsApp", "WeChat", "Line", "Telegram", "Kakao", "Snap", "Discord"
- Payment/account/channel identifiers:
  bank names, card brands, PayPal, Venmo, CashApp, Alipay, WeChat Pay, IBAN,
  sort code, account number, routing number, wallet address, crypto address
- Any explicit call-to-action to message/click/link/pay/subscribe.
WHAT GOES TO RESIDUAL
Everything else that does not match the hard discriminative rules.
OUTPUT FORMAT (MUST MATCH EXACTLY)
Discriminative Text:
<copy lines here exactly>
Residual Text:
<copy remaining lines here exactly>
Remember:
- First line must be exactly: Discriminative Text:
- Then later exactly: Residual Text:
- Output ONLY these two sections, nothing else.
""".strip()

def agent1_process(original_text) -> Dict[str, str]:
    if not isinstance(original_text, str):
        original_text = ""

    full_prompt = AGENT1_PROMPT + "\n\n" + original_text

    try:
        response = local_llama_generate(full_prompt)
    except Exception as e:
        print(f"Agent 1 failed: {e}")
        response = ""

    if not response.strip():
        return {"Discriminative Text": "", "Residual Text": original_text or ""}

    disc_lines, resid_lines = [], []
    current_section = None

    for line in response.splitlines():
        line = line.strip()
        if not line: continue
        if line.startswith("Discriminative Text"):
            current_section = "D"
            continue
        if line.startswith("Residual Text"):
            current_section = "R"
            continue
        if current_section == "D":
            disc_lines.append(line)
        elif current_section == "R":
            resid_lines.append(line)

    return {
        "Discriminative Text": "\n".join(disc_lines),
        "Residual Text": "\n".join(resid_lines)
    }

def run_agent1_and_save_csv(input_csv: str, output_csv: str):
    df = pd.read_csv(input_csv)

    if os.path.exists(output_csv):
        df_out = pd.read_csv(output_csv)
        start_idx = len(df_out)
        print(f"Resuming Agent 1 from index {start_idx}")
    else:
        start_idx = 0

    for i in range(start_idx, len(df)):
        row = df.iloc[i].copy()
        result = agent1_process(row.get("text", ""))
        row["Discriminative Text"] = result["Discriminative Text"]
        row["Residual Text"] = result["Residual Text"]

        pd.DataFrame([row]).to_csv(output_csv, mode='a', header=not os.path.exists(output_csv), index=False)
        if i % 10 == 0:
            print(f"Agent 1: {i}/{len(df)} processed")


# 5) Tavily Search

EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_RE = re.compile(r"(?:(?:\+|00)\d{1,3}[\s-]?)?(?:\(?\d{2,4}\)?[\s-]?)?\d{3,4}[\s-]?\d{3,4}")
HANDLE_RE = re.compile(r"(?<!\w)@[A-Za-z0-9._]{2,30}")
URL_RE = re.compile(r"https?://\S+|\b(?:[A-Za-z0-9-]+\.)+[A-Za-z]{2,}\b")

def extract_search_targets(discriminative_text: str) -> List[str]:
    disc = discriminative_text or ""
    targets = EMAIL_RE.findall(disc)

    for m in PHONE_RE.findall(disc):
        digits = re.sub(r"\D", "", m)
        if 9 <= len(digits) <= 15:
            targets.append(m.strip())

    targets += HANDLE_RE.findall(disc)

    for m in URL_RE.findall(disc):
        m = m.strip().strip(".,;:!()[]{}<>")
        if m and m not in targets:
            targets.append(m)

    seen = set()
    out = []
    for t in targets:
        if t.lower() not in seen:
            seen.add(t.lower())
            out.append(t)
    return out[:6]

def tavily_search(query: str) -> Dict[str, Any]:
    headers = {"Content-Type": "application/json"}
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": 2,
        "search_depth": "basic",
        "include_answer": False,
        "include_raw_content": False
    }
    def api_call():
        resp = requests.post(TAVILY_ENDPOINT, json=payload, headers=headers, timeout=30)
        resp.raise_for_status()
        return resp.json()
    return retry_tavily(api_call)

def tavily_evidence_for_row(row: pd.Series) -> Dict[str, Any]:
    disc = row.get("Discriminative Text", None)

    if not isinstance(disc, str) or not disc.strip() or disc.lower() == "nan":
        return {"targets": [], "queries": [], "results": []}

    targets = extract_search_targets(disc)
    if not targets:
        return {"targets": [], "queries": [], "results": []}

    queries = []
    results_compact = []

    for t in targets:
        q = f"{t} scam OR fraud OR complaint OR review"
        queries.append(q)
        try:
            res = tavily_search(q)
            items = [{"title": i.get("title", ""), "url": i.get("url", ""), "content": i.get("content", "")[:280], "score": i.get("score", None)} for i in (res.get("results") or [])[:2]]
        except:
            items = []

        results_compact.append({"target": t, "query": q, "items": items})
        time.sleep(0.2)

    return {"targets": targets, "queries": queries, "results": results_compact}

def add_tavily_evidence_column(agent1_csv: str, output_csv: str):
    df = pd.read_csv(agent1_csv)

    if os.path.exists(output_csv):
        df_out = pd.read_csv(output_csv)
        start_idx = len(df_out)
    else:
        start_idx = 0

    for i in range(start_idx, len(df)):
        row = df.iloc[i].copy()
        ev = tavily_evidence_for_row(row)
        row["tavily_evidence"] = json.dumps(ev, ensure_ascii=False)
        pd.DataFrame([row]).to_csv(output_csv, mode='a', header=not os.path.exists(output_csv), index=False)
        if i % 10 == 0:
            print(f"Tavily: {i}/{len(df)} processed")


# 6) Agent 2: Binary Classifier

AGENT2_PROMPT = r"""
You are Agent2, a strict binary classifier.
Return ONLY one digit: 0 or 1.

You are given:
- text: full Instagram bio
- Discriminative Text: extracted high-signal cues (may be empty)
- Residual Text: remaining context (low priority)
- tavily_evidence: JSON (may be empty)

GOAL:
Match the dataset labels. In this dataset, many bios that look like creators/influencers
are still label=0. Therefore, you must NOT output 1 based on weak “creator vibes”.


A) PRIORITY & NORMALIZATION

1) Discriminative Text > text > Residual Text.
2) If Discriminative Text is empty, do NOT assume positive. Treat as unknown.
3) tavily_evidence empty is NEUTRAL (not positive, not negative).
4) Ignore corrupted emoji/encoding artifacts like "ðŸ", "ğŸ" etc.


B) CORE DECISION: SCORE + THRESHOLD

Compute a score using ONLY explicit cues in the text.
Default output is 0 unless the score reaches the threshold.

Start score = 0.

Add STRONG POSITIVE signals (+3 each):
+3  Mentions a recognized institution/platform/media network explicitly:
    Examples: "AM 640", "iHeartRadio", "HGTV", "NBC", "radio", "TV presenter"
+3  Clearly indicates an official organization/brand/service venue with operations:
    Examples: salon + booking schedule (W-2-F / appointments / books closed),
    business address + official brand handle, store/clinic/studio name + services.
+3  Clear official topic authority brand like "@SCIENCE" + inquiries email.

Add MODERATE POSITIVE signals (+2 each):
+2  Explicit "Business Inquiries:" / "Press:" / "Media:" + contact email
+2  Two or more consistent external handles pointing to a brand ecosystem
    (e.g., multiple linked brand accounts + role like Founder/Owner)
+2  tavily_evidence contains matching results that align with the bio identity
    (must be consistent; do not assume if results are unclear)

Add WEAK signals (+1 each) [NEVER sufficient alone]:
+1  Has an email
+1  Has "collab", "DM for collabs"
+1  Has "content creator", "influencer", "blogger"
+1  Has TikTok/YouTube mention
+1  Has generic profession label (photographer, stylist, coach) without external verification

Add STRONG NEGATIVE signals (-3 each):
-3  Pure diary / personal vibe only, no operational or institutional cues:
    examples: "visual diary", "my life", random quotes only
-3  Generic lifestyle list only (travel | fashion | beauty | mindset) with no verifiable anchor
-3  Text is mostly hashtags/emojis with no concrete role/entity

Add MODERATE NEGATIVE signals (-2 each):
-2  Overly generic “creator template” bios:
    "lifestyle | travel | fashion | mindset | growth" + DM/email only
-2  Only location + personal descriptors (mom/wife/dog lover) without institution/entity
-2  Discriminative Text contains only social handles or hashtags, no real-world anchor

C) OUTPUT THRESHOLD

If score >= 4  -> output 1
Else           -> output 0

IMPORTANT SAFETY GUARDRAILS:
- Never output 1 if the ONLY evidence is: email + "collab" + "lifestyle/blogger".
  (This pattern is frequently label=0 in this dataset.)
- If uncertain, output 0 (because false positives dominate your current errors).
- Return ONLY 0 or 1. No explanation.



""".strip()

def agent2_judge_from_row(row: pd.Series) -> int:
    disc = row.get("Discriminative Text", "")
    resid = row.get("Residual Text", "")
    tav = row.get("tavily_evidence", "")

    # Default to 0 if no discriminative text is found to reduce false positives
    if not isinstance(disc, str) or not disc.strip() or disc.lower() == "nan":
        return 0

    user_input = f"""
Discriminative Text:\n{disc}

Residual Text:\n{resid}

Tavily evidence JSON:\n{tav}
""".strip()

    resp = local_llama_generate(AGENT2_PROMPT + "\n\n" + user_input)

    try:
        match = re.search(r'\d', resp)
        return int(match.group()) if match else 0
    except Exception:
        return 0

def run_agent2_and_save_csv(agent1_tavily_csv: str, output_csv: str):
    df = pd.read_csv(agent1_tavily_csv)

    if os.path.exists(output_csv):
        df_out = pd.read_csv(output_csv)
        start_idx = len(df_out)
        print(f"Resuming Agent 2 from index {start_idx}")
    else:
        start_idx = 0

    for i in range(start_idx, len(df)):
        row = df.iloc[i].copy()
        pred = agent2_judge_from_row(row)
        row["agent2_pred"] = pred

        pd.DataFrame([row]).to_csv(output_csv, mode='a', header=not os.path.exists(output_csv), index=False)
        if i % 10 == 0:
            print(f"Agent 2: {i}/{len(df)} processed")


# 7) Main Execution

if __name__ == "__main__":
    print("Starting pipeline...")

    input_csv = "/content/instagram_samples500.csv"
    after_a1 = "/content/instagram_after_agent1_local.csv"
    after_tav = "/content/instagram_after_tavily_local.csv"
    final_out = "/content/instagram_final_results_local.csv"

    run_agent1_and_save_csv(input_csv, after_a1)
    print("Agent 1 completed.")

    add_tavily_evidence_column(after_a1, after_tav)
    print("Tavily search completed.")

    run_agent2_and_save_csv(after_tav, final_out)
    print(f"Pipeline finished. Results saved to {final_out}")

Loading local model: meta-llama/Meta-Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded successfully.
Starting pipeline...
Agent 1: 0/500 processed
Agent 1: 10/500 processed
Agent 1: 20/500 processed
Agent 1: 30/500 processed
Agent 1: 40/500 processed
Agent 1: 50/500 processed
Agent 1: 60/500 processed
Agent 1: 70/500 processed
Agent 1: 80/500 processed
Agent 1: 90/500 processed
Agent 1: 100/500 processed
Agent 1: 110/500 processed
Agent 1: 120/500 processed
Agent 1: 130/500 processed
Agent 1: 140/500 processed
Agent 1: 150/500 processed
Agent 1: 160/500 processed
Agent 1: 170/500 processed
Agent 1: 180/500 processed
Agent 1: 190/500 processed
Agent 1: 200/500 processed
Agent 1: 210/500 processed
Agent 1: 220/500 processed
Agent 1: 230/500 processed
Agent 1: 240/500 processed
Agent 1: 250/500 processed
Agent 1: 260/500 processed
Agent 1: 270/500 processed
Agent 1: 280/500 processed
Agent 1: 290/500 processed
Agent 1: 300/500 processed
Agent 1: 310/500 processed
Agent 1: 320/500 processed
Agent 1: 330/500 processed
Agent 1: 340/500 processed
Agent 1: 350/500 p

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df_result = pd.read_csv("/content/instagram_final_results_local.csv")

y_true = df_result["label"]
y_pred = df_result["agent2_pred"]

# Filter out rows with missing labels or predictions
mask = y_true.notna() & y_pred.notna()

y_true_clean = y_true[mask].tolist()
y_pred_clean = y_pred[mask].tolist()

# Enforce strict binary labels
# Note: The 1B model occasionally hallucinates and outputs other digits (e.g., 2, 3, 4).
# We treat anything that isn't strictly '1' as a '0' (negative class).
y_pred_clean = [1 if str(val).strip() in ["1", "1.0"] else 0 for val in y_pred_clean]
y_true_clean = [1 if str(val).strip() in ["1", "1.0"] else 0 for val in y_true_clean]

# Calculate standard binary classification metrics
acc = accuracy_score(y_true_clean, y_pred_clean)
p = precision_score(y_true_clean, y_pred_clean, average="binary")
r = recall_score(y_true_clean, y_pred_clean, average="binary")
f1 = f1_score(y_true_clean, y_pred_clean, average="binary")

# Calculate macro F1 for additional reference
f1_macro = f1_score(y_true_clean, y_pred_clean, average="macro")

print(f"Used rows: {mask.sum()} / {len(df_result)}")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {p:.4f}")
print(f"Recall:    {r:.4f}")
print(f"F1:        {f1:.4f}")
print(f"F1-macro:  {f1_macro:.4f}")

Used rows: 500 / 500
Accuracy:  0.6300
Precision: 0.1667
Recall:    0.0055
F1:        0.0107
F1-macro:  0.3916
